<a href="https://colab.research.google.com/github/ankurdev1-drth/Deep_Learning-/blob/main/02_Training_Deep_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [1]:
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import torch

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5  # kaiming init (or 3 ** 0.5 for LeCun)
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
layer.weight.shape

torch.Size([10, 40])

In [4]:
layer.bias.shape

torch.Size([10])

In [5]:
# another way to for initialization
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

This is clearer and less error prone as compared to the first approach

For applying initialization to the weights of every `nn.Linear` layer in a model the simples option is to write a little function that takes a module, checks whether it's an instance of the `nn.Linear` class, and if so , applies the desired initialization function to its weights. and the function can be then applied using the `apply()` method

In [6]:
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

module = nn.Sequential(
    nn.Linear(40, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
    nn.ReLU()
    )
module.apply(use_he_init)

Sequential(
  (0): Linear(in_features=40, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=1, bias=True)
  (3): ReLU()
)

### Leaky ReLU

In [7]:
alpha = 0.2
model = nn.Sequential(
    nn.Linear(50, 40), #0
    nn.LeakyReLU(negative_slope=alpha), #1
    nn.Linear(40, 20), #2
    nn.LeakyReLU(negative_slope=alpha) #3

)
nn.init.kaiming_uniform_(model[0].weight, alpha,
nonlinearity="leaky_relu")
nn.init.kaiming_uniform_(model[2].weight, alpha,nonlinearity="leaky_relu")

Parameter containing:
tensor([[ 0.0712, -0.0897, -0.2638, -0.0860, -0.3468, -0.1989,  0.2383,  0.0744,
          0.2910, -0.2538, -0.0750, -0.2855, -0.3247,  0.2507,  0.2567,  0.2082,
         -0.3563, -0.3047, -0.1566, -0.0810,  0.2697,  0.1499, -0.1583, -0.2878,
          0.1757,  0.0871, -0.2330, -0.0343,  0.2204, -0.3049, -0.0981,  0.3252,
          0.1955,  0.0615,  0.2694, -0.0394,  0.3340, -0.1514, -0.2360, -0.2498],
        [-0.2432, -0.2486,  0.2930,  0.2879,  0.3354, -0.0801, -0.0876,  0.1453,
         -0.1900, -0.1163, -0.2924, -0.0056,  0.1676, -0.0800, -0.3624, -0.3493,
          0.3472, -0.2089, -0.2716,  0.1119, -0.0081,  0.2791, -0.2895,  0.0625,
          0.2191, -0.2881, -0.0608, -0.2674,  0.0618, -0.2609,  0.1799, -0.3218,
          0.0112,  0.1658,  0.2775, -0.1803, -0.3551, -0.2662,  0.3626, -0.0311],
        [-0.0120, -0.3383,  0.2706, -0.0568, -0.0607,  0.1128, -0.1108, -0.0132,
          0.1785,  0.1456,  0.3696, -0.1868, -0.3760,  0.3748, -0.0536,  0.1571,
    

### Batch Normalization

In [8]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28,  300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.BatchNorm1d(100),
    nn.Linear(100, 10)
)

In [9]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [10]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [11]:
dict(model[1].named_children()).keys()

dict_keys([])

In [12]:
dict(model[1].named_modules()).keys()

dict_keys([''])

Note:
- if BN layers before the activation functions, we can also remove the bias term from the previous `nn.Linear` layers by setting the `bias` hyperparameter to `False`.

In [13]:
Model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

### Layer Normalization

In [14]:
inputs = torch.randn(32, 3, 100, 200)   # batch of random RGB images
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)
result

tensor([[[[ 5.0271e-02,  5.7603e-01,  8.4280e-02,  ...,  5.4032e-01,
           -3.4779e-01,  1.0387e+00],
          [ 2.5925e-01, -1.3737e+00,  3.0384e-01,  ...,  9.3424e-01,
           -1.2317e+00, -2.0864e-01],
          [-5.8602e-02, -1.6581e+00,  8.3240e-02,  ..., -2.8037e-02,
           -7.0846e-01,  2.1782e-01],
          ...,
          [ 1.3057e-01,  9.5036e-01, -2.4901e-01,  ...,  7.9101e-01,
            8.2319e-01, -4.2525e-01],
          [ 3.6864e-01, -7.3771e-01,  1.4725e+00,  ...,  1.7190e+00,
            1.1707e-01, -1.6486e-01],
          [-5.3767e-01, -1.2421e+00, -9.6030e-01,  ...,  8.1784e-01,
           -5.4208e-01,  7.9651e-01]],

         [[-6.2042e-01, -1.0776e+00,  1.5204e+00,  ..., -3.1420e-01,
           -2.9858e-01, -5.1056e-01],
          [ 3.9593e-01,  5.5183e-01, -1.0197e-01,  ..., -4.7733e-01,
           -4.5365e-01,  7.4152e-01],
          [ 1.9406e+00,  3.8890e-01, -1.5430e+00,  ..., -1.7009e+00,
           -3.8694e-01,  6.9611e-02],
          ...,
     

In [15]:
# method 2 to perform the same thing as above!
means = inputs.mean(dim=[2,3], keepdim=True)  #shape [32, 3, 1, 1]
vars_ = inputs.var(dim=[2,3], keepdim=True, unbiased=False) # shape: same
stds = torch.sqrt(vars_ + layer_norm.eps) # eps is a smoothing term (1e-5)
result = layer_norm.weight * (inputs - means) / stds + layer_norm.bias

In [16]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)

In [17]:
result

tensor([[[[ 5.1211e-02,  5.7493e-01,  8.5089e-02,  ...,  5.3936e-01,
           -3.4530e-01,  1.0358e+00],
          [ 2.5938e-01, -1.3673e+00,  3.0380e-01,  ...,  9.3175e-01,
           -1.2258e+00, -2.0670e-01],
          [-5.7239e-02, -1.6505e+00,  8.4052e-02,  ..., -2.6793e-02,
           -7.0458e-01,  2.1811e-01],
          ...,
          [ 1.3120e-01,  9.4781e-01, -2.4691e-01,  ...,  7.8908e-01,
            8.2113e-01, -4.2247e-01],
          [ 3.6834e-01, -7.3371e-01,  1.4679e+00,  ...,  1.7135e+00,
            1.1775e-01, -1.6309e-01],
          [-5.3444e-01, -1.2361e+00, -9.5543e-01,  ...,  8.1580e-01,
           -5.3884e-01,  7.9455e-01]],

         [[-6.3196e-01, -1.0901e+00,  1.5134e+00,  ..., -3.2510e-01,
           -3.0944e-01, -5.2187e-01],
          [ 3.8653e-01,  5.4276e-01, -1.1241e-01,  ..., -4.8857e-01,
           -4.6484e-01,  7.3285e-01],
          [ 1.9345e+00,  3.7949e-01, -1.5565e+00,  ..., -1.7147e+00,
           -3.9799e-01,  5.9527e-02],
          ...,
     

## Gradient Clipping

See the line nn.utils.clip_grad_norm_(...) in the training function in the next section.

# Reusing Pretrained Layers

### Transfer Learning with PyTorch

- X_train_A = images of all items except for T-shirts/tops and pullovers (classes 0 and 2 )
- X_train_B = first 20 images of Tshirt and Pullovers

The validation set and the test set are also split this way, but without restricting the number of images.

We will train a model on set A (classification task with 8 classes), and try to reuse it to tackle set B (binary classification). We hope to transfer a little bit of knowledge from task A to task B, since classes in set A (trousers, dresses, coats, sandals, shirts, sneakers, bags, and ankle boots) are somewhat similar to classes in set B (T-shirts/tops and pullovers). However, since we are using Linear layers, only patterns that occur at the same location can be reused (in contrast, convolutional layers will transfer much better, since learned patterns can be detected anywhere on the image)


In [18]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_openml


fashion_mnist = fetch_openml(name="Fashion-MNIST", as_frame=False) #as_frames = False for getting the numpy arrays instead of pandas dataframe/series
X = torch.FloatTensor(fashion_mnist.data.reshape(-1, 1, 28, 28) / 255.)
y = torch.from_numpy(fashion_mnist.target.astype(int))
in_B = (y == 0) | (y == 2)  # Pullover or T-shirt/top
X_A, y_A = X[~in_B], y[~in_B]
y_A = torch.maximum(y_A - 2, torch.tensor(0))  # [1,3,4,5,6,7,8,9] => [0,..,7]
X_B, y_B = X[in_B], (y[in_B] == 2).to(dtype=torch.float32).view(-1, 1)

train_set_A = TensorDataset(X_A[:-7_000], y_A[:-7000])
valid_set_A = TensorDataset(X_A[-7_000:-5000], y_A[-7000:-5000])
test_set_A = TensorDataset(X_A[-5_000:], y_A[-5000:])
train_set_B = TensorDataset(X_B[:20], y_B[:20])
valid_set_B = TensorDataset(X_B[20:5000], y_B[20:5000])
test_set_B = TensorDataset(X_B[5_000:], y_B[5000:])

train_loader_A = DataLoader(train_set_A, batch_size=32, shuffle=True)
valid_loader_A = DataLoader(valid_set_A, batch_size=32)
test_loader_A = DataLoader(test_set_A, batch_size=32)
train_loader_B = DataLoader(train_set_B, batch_size=32, shuffle=True)
valid_loader_B = DataLoader(valid_set_B, batch_size=32)
test_loader_B = DataLoader(test_set_B, batch_size=32)

In [19]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device


'cpu'

In [20]:
%pip install torchmetrics


import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            # Uncomment to activate gradient clipping:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history



In [21]:
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 8)
  )
model_A = model_A.to(device)

In [22]:
model_A.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=8, bias=True)
)

In [23]:
n_epochs = 20
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.005)
loss_fn = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train(model_A, optimizer, loss_fn, accuracy, train_loader_A, valid_loader_A, n_epochs )

Epoch 1/20, train loss: 0.8492, train metric: 0.7197, valid metric: 0.8120
Epoch 2/20, train loss: 0.4533, train metric: 0.8487, valid metric: 0.8620
Epoch 3/20, train loss: 0.3771, train metric: 0.8727, valid metric: 0.8685
Epoch 4/20, train loss: 0.3405, train metric: 0.8828, valid metric: 0.8890
Epoch 5/20, train loss: 0.3184, train metric: 0.8904, valid metric: 0.8825
Epoch 6/20, train loss: 0.3033, train metric: 0.8953, valid metric: 0.8875
Epoch 7/20, train loss: 0.2911, train metric: 0.9000, valid metric: 0.8900
Epoch 8/20, train loss: 0.2818, train metric: 0.9031, valid metric: 0.8960
Epoch 9/20, train loss: 0.2737, train metric: 0.9053, valid metric: 0.8985
Epoch 10/20, train loss: 0.2668, train metric: 0.9082, valid metric: 0.8900
Epoch 11/20, train loss: 0.2611, train metric: 0.9105, valid metric: 0.9035
Epoch 12/20, train loss: 0.2547, train metric: 0.9125, valid metric: 0.8985
Epoch 13/20, train loss: 0.2502, train metric: 0.9145, valid metric: 0.8855
Epoch 14/20, train lo

In [24]:
torch.manual_seed(42)
model_B = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1)
).to(device)

In [25]:
model_B.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=100, bias=True)
  (2): ReLU()
  (3): Linear(in_features=100, out_features=100, bias=True)
  (4): ReLU()
  (5): Linear(in_features=100, out_features=100, bias=True)
  (6): ReLU()
  (7): Linear(in_features=100, out_features=1, bias=True)
)

In [26]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B, optimizer, loss_fn, accuracy, train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20, train loss: 0.7110, train metric: 0.5000, valid metric: 0.4914
Epoch 2/20, train loss: 0.7083, train metric: 0.5000, valid metric: 0.4954
Epoch 3/20, train loss: 0.7055, train metric: 0.5000, valid metric: 0.4976
Epoch 4/20, train loss: 0.7028, train metric: 0.5000, valid metric: 0.5008
Epoch 5/20, train loss: 0.7001, train metric: 0.5000, valid metric: 0.5034
Epoch 6/20, train loss: 0.6974, train metric: 0.5000, valid metric: 0.5080
Epoch 7/20, train loss: 0.6947, train metric: 0.5000, valid metric: 0.5118
Epoch 8/20, train loss: 0.6920, train metric: 0.5000, valid metric: 0.5159
Epoch 9/20, train loss: 0.6893, train metric: 0.5000, valid metric: 0.5197
Epoch 10/20, train loss: 0.6866, train metric: 0.5000, valid metric: 0.5225
Epoch 11/20, train loss: 0.6840, train metric: 0.5000, valid metric: 0.5263
Epoch 12/20, train loss: 0.6813, train metric: 0.5000, valid metric: 0.5311
Epoch 13/20, train loss: 0.6787, train metric: 0.5000, valid metric: 0.5339
Epoch 14/20, train lo

In [27]:
# reusing all the layers except the output layer
import copy

torch.manual_seed(42)
reused_layer = copy.deepcopy(model_A[: -1])
model_B_on_A = nn.Sequential(
    *reused_layer,
    nn.Linear(100, 1) # new output layer for task B
).to(device)

Note:
- `copy.deepcopy()` function to copy all the modules and submodules in `nn.Sequential` module
- model_B_on_A is another `nn.Sequential` module based on the reused layers of model A plus a new output layer for task B it has a single output since task B is binary classification.

In [28]:
# freezing the reused layers during the first few epochs:
for layer in model_B_on_A[:-1]:
  for param in layer.parameters():
    param.requires_grad = False


In [29]:
n_epochs = 10
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

Epoch 1/10, train loss: 0.6149, train metric: 0.4000, valid metric: 0.5402
Epoch 2/10, train loss: 0.5850, train metric: 0.4500, valid metric: 0.5783
Epoch 3/10, train loss: 0.5589, train metric: 0.6500, valid metric: 0.6410
Epoch 4/10, train loss: 0.5367, train metric: 0.8000, valid metric: 0.7179
Epoch 5/10, train loss: 0.5185, train metric: 0.8500, valid metric: 0.7855
Epoch 6/10, train loss: 0.5041, train metric: 0.8500, valid metric: 0.8367
Epoch 7/10, train loss: 0.4933, train metric: 0.9000, valid metric: 0.8663
Epoch 8/10, train loss: 0.4851, train metric: 0.9000, valid metric: 0.8787
Epoch 9/10, train loss: 0.4783, train metric: 0.8500, valid metric: 0.8855
Epoch 10/10, train loss: 0.4718, train metric: 0.8500, valid metric: 0.8898


In [30]:
# unfreezing the reused layers during the first few epochs:
for layer in model_B_on_A[2:]:
  for param in layer.parameters():
    param.requires_grad = True

In [31]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.01)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20, train loss: 0.4654, train metric: 0.8500, valid metric: 0.8972
Epoch 2/20, train loss: 0.4511, train metric: 0.9000, valid metric: 0.9022
Epoch 3/20, train loss: 0.4372, train metric: 0.9000, valid metric: 0.9074
Epoch 4/20, train loss: 0.4238, train metric: 0.9500, valid metric: 0.9112
Epoch 5/20, train loss: 0.4108, train metric: 0.9500, valid metric: 0.9139
Epoch 6/20, train loss: 0.3983, train metric: 0.9500, valid metric: 0.9173
Epoch 7/20, train loss: 0.3861, train metric: 0.9500, valid metric: 0.9179
Epoch 8/20, train loss: 0.3744, train metric: 0.9500, valid metric: 0.9193
Epoch 9/20, train loss: 0.3631, train metric: 0.9500, valid metric: 0.9213
Epoch 10/20, train loss: 0.3521, train metric: 0.9500, valid metric: 0.9221
Epoch 11/20, train loss: 0.3415, train metric: 0.9500, valid metric: 0.9233
Epoch 12/20, train loss: 0.3312, train metric: 0.9500, valid metric: 0.9249
Epoch 13/20, train loss: 0.3212, train metric: 0.9500, valid metric: 0.9259
Epoch 14/20, train lo

In [32]:
evaluate_tm(model_B_on_A, test_loader_B, accuracy)

tensor(0.9360)

## Faster Optimizers

In [33]:

def build_model(seed=43):
  torch.manual_seed(seed)
  model = nn.Sequential(
      nn.Flatten(),
      nn.Linear(28 * 28, 100),
      nn.ReLU(),
      nn.Linear(100, 100),
      nn.ReLU(),
      nn.Linear(100, 100),
      nn.ReLU(),
      nn.Linear(100, 10)
  ).to(device)
  model.apply(use_he_init)
  return model

def test_optimizer(model, optimizer, n_epochs=10, batch_size=32):
  train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
  valid_loader = DataLoader(valid_set, batch_size=batch_size)
  test_loader = DataLoader(test_set, batch_size=32)
  xentropy = nn.CrossEntropyLoss()
  accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
  history = train(model, optimizer, xentropy, accuracy.to(device),
                  train_loader, valid_loader, n_epochs)
  return history, evaluate_tm(model, test_loader, accuracy)


In [34]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

## Momentum Optimization

In [40]:
train_set = TensorDataset(X[:55_000], y[:55_000])
valid_set = TensorDataset(X[55_000:60_000], y[55_000:60_000])
test_set = TensorDataset(X[60_000:], y[60_000:])

model = build_model()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.01)
history_momentum, accuracy_momentum = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.5822, train metric: 0.7977, valid metric: 0.8336
Epoch 2/10, train loss: 0.4185, train metric: 0.8500, valid metric: 0.8590
Epoch 3/10, train loss: 0.3763, train metric: 0.8646, valid metric: 0.8680
Epoch 4/10, train loss: 0.3474, train metric: 0.8736, valid metric: 0.8686
Epoch 5/10, train loss: 0.3296, train metric: 0.8799, valid metric: 0.8730
Epoch 6/10, train loss: 0.3120, train metric: 0.8862, valid metric: 0.8682
Epoch 7/10, train loss: 0.3004, train metric: 0.8888, valid metric: 0.8782
Epoch 8/10, train loss: 0.2883, train metric: 0.8927, valid metric: 0.8582
Epoch 9/10, train loss: 0.2800, train metric: 0.8967, valid metric: 0.8796
Epoch 10/10, train loss: 0.2700, train metric: 0.9018, valid metric: 0.8718


### Nesterov Accelerated Gradient

In [41]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, nesterov=True, lr=0.01)
history_nesterov, accuracy_nesterov = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.5651, train metric: 0.8047, valid metric: 0.8406
Epoch 2/10, train loss: 0.4168, train metric: 0.8525, valid metric: 0.8582
Epoch 3/10, train loss: 0.3767, train metric: 0.8650, valid metric: 0.8712
Epoch 4/10, train loss: 0.3482, train metric: 0.8737, valid metric: 0.8720
Epoch 5/10, train loss: 0.3301, train metric: 0.8807, valid metric: 0.8734
Epoch 6/10, train loss: 0.3126, train metric: 0.8863, valid metric: 0.8700
Epoch 7/10, train loss: 0.2993, train metric: 0.8895, valid metric: 0.8772
Epoch 8/10, train loss: 0.2886, train metric: 0.8938, valid metric: 0.8786
Epoch 9/10, train loss: 0.2795, train metric: 0.8974, valid metric: 0.8802
Epoch 10/10, train loss: 0.2694, train metric: 0.9020, valid metric: 0.8706


## AdaGrad

In [44]:
model = build_model()
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01)
history_adagrad, accuracy_adagrad = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.8735, train metric: 0.7198, valid metric: 0.8038
Epoch 2/10, train loss: 0.5652, train metric: 0.8059, valid metric: 0.8140
Epoch 3/10, train loss: 0.5089, train metric: 0.8225, valid metric: 0.8260
Epoch 4/10, train loss: 0.4781, train metric: 0.8324, valid metric: 0.8330
Epoch 5/10, train loss: 0.4576, train metric: 0.8405, valid metric: 0.8400
Epoch 6/10, train loss: 0.4423, train metric: 0.8460, valid metric: 0.8420
Epoch 7/10, train loss: 0.4298, train metric: 0.8511, valid metric: 0.8436
Epoch 8/10, train loss: 0.4199, train metric: 0.8544, valid metric: 0.8516
Epoch 9/10, train loss: 0.4105, train metric: 0.8573, valid metric: 0.8532
Epoch 10/10, train loss: 0.4027, train metric: 0.8600, valid metric: 0.8558


## RMSProp

In [45]:
model = build_model()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.01, alpha=0.9)
history_rmsprop, accuracy_rmsprop = test_optimizer(model, optimizer)

Epoch 1/10, train loss: 0.7701, train metric: 0.7619, valid metric: 0.8212
Epoch 2/10, train loss: 0.6423, train metric: 0.8088, valid metric: 0.8042
Epoch 3/10, train loss: 0.6736, train metric: 0.8131, valid metric: 0.8208
Epoch 4/10, train loss: 0.7114, train metric: 0.8153, valid metric: 0.8012
Epoch 5/10, train loss: 0.7223, train metric: 0.8159, valid metric: 0.8338
Epoch 6/10, train loss: 0.7973, train metric: 0.8157, valid metric: 0.8204
Epoch 7/10, train loss: 0.8257, train metric: 0.8193, valid metric: 0.8176
Epoch 8/10, train loss: 0.8254, train metric: 0.8150, valid metric: 0.8068
Epoch 9/10, train loss: 0.8474, train metric: 0.8128, valid metric: 0.8242
Epoch 10/10, train loss: 0.9292, train metric: 0.8146, valid metric: 0.7750
